# Automatic verification how close our PF to the bruteforced one

In [1]:
from glob import glob
import pandas as pd

list_dataset = ["Lastfm", "Amazon-lb", "QK-video", "Jester", "ML-10M", "ML-20M"]

In [2]:
from pymoo.indicators.gd import GD

In [ ]:
result = dict()
non_zero_GD_bruteforce = dict()
non_zero_GD_paretosmall = dict()

for dataset in list_dataset:
    pareto_small_files = glob(f"brute_force/pareto_small_{dataset}_*_oraclefair_at3.pickle")
    bruteforce_files = glob(f"brute_force/scores*_{dataset}_*.pkl")

    print(dataset)

    result[dataset] = dict()
    non_zero_GD_bruteforce[dataset] = dict()
    non_zero_GD_paretosmall[dataset] = dict()

    for file in bruteforce_files:
        sample = file\
                    .rsplit("_", maxsplit=1)[1]\
                    .replace(".pkl","")
        
        matching_pareto_small_files = list(filter(lambda x: f"_{sample}_" in x, pareto_small_files))

        assert len(matching_pareto_small_files) == 1

        pareto_small_file = matching_pareto_small_files[0]


        points = pd.read_pickle(file)
        df_points = pd.DataFrame(points)

        best_fair_for_given_rel = df_points\
                                        .groupby("ndcg@3")\
                                        .max()\
                                        .reset_index()
        best_rel_for_given_fair = df_points\
                                        .groupby("Ent_our@3")\
                                        .max()\
                                        .reset_index()
        brute_force_pareto = best_fair_for_given_rel.merge(best_rel_for_given_fair, how="inner")

        pareto_points = pd.read_pickle(pareto_small_file)
        df_pareto_points = pd.DataFrame(pareto_points)

        # remove dominated solutions
        df_pareto_points = df_pareto_points\
                                        .sort_values("ndcg@3", ascending=False)\
                                        .drop_duplicates("ndcg@3", keep="last")
        # we keep last because fairness would be higher at the last index

        # Compute Generational Distance between brute_force_pareto and our algo's pareto 
        ind = GD(brute_force_pareto.values)
        generational_distance = ind(df_pareto_points.values).item()

        result[dataset][int(sample)] = generational_distance

        if generational_distance !=0:
            print(sample, generational_distance)

            # save our/bruteforce PF to a dict too for easy lookup
            non_zero_GD_bruteforce[dataset][int(sample)] = brute_force_pareto
            non_zero_GD_paretosmall[dataset][int(sample)] = df_pareto_points


In [9]:
num_non_zero_GD = dict()

for this_data, this_dict in non_zero_GD_paretosmall.items():
    num_non_zero_GD[this_data] = len(this_dict)

In [7]:
df_result = pd.DataFrame(result)\
                            .mean()\
                            .round(3)\
                            .to_frame()\
                            .rename(columns={0:"Average GD"})

In [11]:
df_result["% Zero GD"] = 100 - pd.Series(num_non_zero_GD)

In [12]:
df_result

,Average GD,% Zero GD
Lastfm,0.000,100
Amazon-lb,0.003,93
QK-video,0.000,100
Jester,0.002,97
ML-10M,0.000,100
ML-20M,0.001,98
